<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_11_Strategic_Caching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from functools import lru_cache
import requests

@lru_cache(maxsize=500)     # cache up to 500 recent results
def get_user_name(user_id: int) -> str:
    """Fetch user name from JSONPlaceholder API."""
    # visible on cache miss
    print(f"Fetching user {user_id} from API...")

    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    user_data = response.json()

    return user_data.get('name', 'Unknown')

In [ ]:
# First call → cache miss (actual API call happens)
print(get_user_name(4))

# Second call → cache hit (returns instantly from cache)
print(get_user_name(4))

Fetching user 4 from API...
Patricia Lebsack
Patricia Lebsack


In [ ]:
get_user_name.cache_info()

CacheInfo(hits=1, misses=1, maxsize=500, currsize=1)

## **Caching in a pipeline**

In [ ]:
import pandas as pd

df = pd.DataFrame({"user_id": [1, 2, 4, 4, 5, 6]})
df.head()

,user_id
0,1
1,2
2,4
3,4
4,5


In [ ]:

from functools import lru_cache
import requests

@lru_cache(maxsize=500)     # cache up to 500 recent results
def get_user_name(user_id: int) -> str:
    """Fetch user name from JSONPlaceholder API."""
    print(f"Fetching user {user_id} from API...")   # visible on cache miss

    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    user_data = response.json()

    return user_data.get('name', 'Unknown')


def enrich_with_user_name(df: pd.DataFrame) -> pd.DataFrame:
    """Enrich transaction data with user names using cached lookup."""
    df = df.copy()

    # With @lru_cache, repeated user_ids are served instantly from cache
    df['user_name'] = df['user_id'].map(get_user_name)
    return df

enrich_with_user_name(df)

Fetching user 1 from API...
Fetching user 2 from API...
Fetching user 4 from API...
Fetching user 5 from API...
Fetching user 6 from API...


,user_id,user_name
0,1,Leanne Graham
1,2,Ervin Howell
2,4,Patricia Lebsack
3,4,Patricia Lebsack
4,5,Chelsey Dietrich
5,6,Mrs. Dennis Schulist


## **Custom caching**

In [ ]:
from functools import wraps
import time
import requests

def ttl_cache(ttl_seconds=5):
    def decorator(func):
        cache = {}

        @wraps(func)
        def wrapper(user_id):
            current_time = time.time()

            # Check if cached and still valid
            if user_id in cache:
                result, timestamp = cache[user_id]
                if current_time - timestamp < ttl_seconds:
                    return result  # cache hit (still fresh)

            # Cache miss or expired
            print(f"Fetching user {user_id} from API...")
            result = func(user_id)
            cache[user_id] = (result, current_time)
            return result

        return wrapper
    return decorator


@ttl_cache(ttl_seconds=5)
def get_user_name(user_id: int) -> str:
    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    return response.json().get('name', 'Unknown')

# Simulating pipeline enrichment
user_ids = [4, 4]

# Run 1: First request hits API; second request hits cache
print([get_user_name(uid) for uid in user_ids])
time.sleep(6)  # let cache expire
# Run 2: TTL has expired; data is fetched fresh from API again
print([get_user_name(uid) for uid in user_ids])

Fetching user 4 from API...
['Patricia Lebsack', 'Patricia Lebsack']
Fetching user 4 from API...
['Patricia Lebsack', 'Patricia Lebsack']
